# Project : 서울시 빅데이터 활용 경진대회
- 프로젝트 목적 : 서울시 공공 데이터를 바탕으로 N년간의 물품 가격의 시계열 패턴, 지역별 차이, 외부 요인에 의한 영향 등을 정량화하고 시민 체감형 물가 현황 대시보드를 제작
- 데이터 정보
    - 규모 : 319,296 rows * 19 columns
    - 기간 : 2023-01-20 ~ 2026-04-14
    - 출처 : [서울 데이터 허브](https://data.seoul.go.kr/bsp/wgs/dataView/data300View/10003.do)
- 최종 수정 : 2026-05-01

## 라이브러리 import

In [377]:
# 데이터 처리
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from datetime import datetime
import re
import math

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
# import missingno as msno  # 결측치 분포 시각화
## 한글 깨짐 방지
# 윈도우
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# 맥
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 판다스 출력 옵션 설정 (데이터가 많을 때 생략되는 것을 방지)
pd.set_option('display.max_columns', None)  # 모든 컬럼 출력
pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.float_format', '{:.4f}'.format) # 소수점 4자리까지 고정 (가독성 향상)

# 그 외
# 경고 무시
import warnings
warnings.filterwarnings("ignore")

# 시스템
import os
import gc

## 데이터 로드 & 탐색

In [378]:
PATH = '/Users/hyun/Documents/project/seoul-market-price-analysis/data/raw/'

df25 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2025년이후).csv', header=1, low_memory=False) # 모든 데이터를 문자로 읽어옴, 메모리 경고를 끄고 한 번에 읽어서 타입을 추론
df24 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2024년).csv', low_memory=False, encoding='cp949')
df23 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2023년).csv', low_memory=False, encoding='cp949')

df25.shape, df24.shape, df23.shape

((144684, 19), (98053, 14), (76559, 14))

## preprocessing & feature engineering

### 데이터셋 통합 전

#### 칼럼명 통일

In [379]:
col_mapping = {
    '일련번호':                         'sn',
    '시장/마트 번호':                   'mkplc_mart_no',
    '시장/마트 이름':                   'mkplc_mart_nm',
    '품목 번호':                        'prdlst_no',
    '품목 이름':                        'prdlst_nm',
    '실판매규격':                       'real_sle_stndrd',
    '가격(원)':                         'pc',
    '년도-월':                          'ym',
    '비고':                             'rmrk',
    '시장유형 구분(시장/마트) 코드':    'mkplc_type_cd',
    '시장유형 구분(시장/마트) 이름':    'mkplc_type_nm',
    '자치구 코드':                      'atdrc_cd',
    '자치구 이름':                      'atdrc',
    '점검일자':                         'chck_ymd',
}

df23.rename(columns=col_mapping, inplace=True)
df24.rename(columns=col_mapping, inplace=True)

print("✅ 칼럼명 통일 완료")

✅ 칼럼명 통일 완료


#### 칼럼별 데이터타입 통일

In [380]:
# ym 칼럼 포맷 통일
# df23 : '2023-02' 형식
df23['ym'] = pd.to_datetime(df23['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')
# df24 : 'Jan-24' 형식
df24['ym'] = pd.to_datetime(df24['ym'], format='%b-%y', errors='coerce').dt.strftime('%Y-%m')
# df25 : '2025-01' 형식
df25['ym'] = pd.to_datetime(df25['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')

# 2023-01 제거 (98건, 2월 누락으로 연속성 없음)
df23 = df23[df23['ym'] != '2023-01'].copy()

# mkplc_type_cd 타입 통일
for df in [df23, df24, df25]:
    df['mkplc_type_cd'] = df['mkplc_type_cd'].astype('Int64')

print("✅ ym 포맷 통일 완료")
print(f"  df23: {df23['ym'].min()} ~ {df23['ym'].max()}")
print(f"  df24: {df24['ym'].min()} ~ {df24['ym'].max()}")
print(f"  df25: {df25['ym'].min()} ~ {df25['ym'].max()}")

✅ ym 포맷 통일 완료
  df23: 2023-03 ~ 2023-12
  df24: 2024-01 ~ 2024-12
  df25: 2025-01 ~ 2026-04


#### 품목명 파싱으로 대표품목명 변수 생성

In [381]:
def parse_item_details(text):
    """prdlst_nm → (핵심품목명, 품종, 규격) 분해"""
    if pd.isna(text):
        return '기타', '기본', '규격없음'

    text = str(text)

    # 품종/산지: 괄호 안 텍스트
    v_match = re.search(r'\((.*?)\)', text)
    variety = v_match.group(1).strip() if v_match else '기본'

    # 규격: 숫자 + 단위
    s_match = re.search(r'(\d+(?:\.\d+)?\s*[a-zA-Z가-힣]+)', text)
    spec = s_match.group(1).strip() if s_match else '규격없음'

    # 핵심 품목명: 괄호·숫자·단위·특수문자 제거
    core = re.sub(r'\(.*?\)', '', text)
    core = re.sub(r'\d+(?:\.\d+)?\s*[a-zA-Z가-힣]*', '', core)
    core = re.sub(r'[.,/]', '', core).strip()
    core = ' '.join(core.split())

    return core, variety, spec

for name, df in [('df23', df23), ('df24', df24), ('df25', df25)]:
    parsed = df['prdlst_nm'].apply(parse_item_details)
    df['new_std_nm'] = [x[0] for x in parsed]
    df['variety']    = [x[1] for x in parsed]
    df['spec']       = [x[2] for x in parsed]
    print(f"  {name} 파싱 완료 → 고유 품목 수: {df['new_std_nm'].nunique()}개")

print("✅ 품목명 파싱 완료")

  df23 파싱 완료 → 고유 품목 수: 103개
  df24 파싱 완료 → 고유 품목 수: 99개
  df25 파싱 완료 → 고유 품목 수: 106개
✅ 품목명 파싱 완료


In [382]:
ori_all_unique_set = set(df23['prdlst_nm']) | set(df24['prdlst_nm']) | set(df25['prdlst_nm'])
print(f"  전체 3개년도 고유 품목 수: {len(ori_all_unique_set)}개")

  전체 3개년도 고유 품목 수: 205개


In [383]:
all_unique_set = set(df23['new_std_nm']) | set(df24['new_std_nm']) | set(df25['new_std_nm'])
print(f"  전체 3개년도 고유 대표품목 수: {len(all_unique_set)}개")

  전체 3개년도 고유 대표품목 수: 128개


#### 품목번호 맵핑오류 처리

In [384]:
# 임계값 설정
THRESHOLD = 0.05  # 5% 미만이면 오입력으로 판단

def detect_mapping_errors(df, year_label, threshold=THRESHOLD):
    df = df.copy()

    # 결측치 처리 ('기타' → prdlst_no 기준 최빈 new_std_nm으로 대체)
    mode_map = (
        df[df['new_std_nm'] != '기타']
        .groupby('prdlst_no')['new_std_nm']
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )

    null_before = (df['new_std_nm'] == '기타').sum()
    df['new_std_nm'] = df.apply(
        lambda row: mode_map.get(row['prdlst_no'], '기타')
        if row['new_std_nm'] == '기타' else row['new_std_nm'],
        axis=1
    )
    null_after = (df['new_std_nm'] == '기타').sum()
    print(f"\n[{year_label}]")
    print(f"  결측치(기타) 처리: {null_before:,}건 → {null_after:,}건")

    # 오입력 탐지
    total_counts  = df.groupby('prdlst_no')['new_std_nm'].transform('count')
    mode_per_code = df.groupby('prdlst_no')['new_std_nm'].transform(
        lambda x: x.value_counts().index[0]
    )
    item_counts = df.groupby(['prdlst_no', 'new_std_nm'])['new_std_nm'].transform('count')

    df['_ratio']     = item_counts / total_counts
    df['_mode_item'] = mode_per_code
    df['_is_wrong']  = (
        (df['new_std_nm'] != df['_mode_item']) &
        (df['_ratio'] < threshold)
    )

    # 소멸 품목 보호
    wrong_items   = set(df[df['_is_wrong']]['new_std_nm'])
    correct_items = set(df[~df['_is_wrong']]['new_std_nm'])
    vanishing     = wrong_items - correct_items

    if vanishing:
        print(f"  소멸 보호 품목: {sorted(vanishing)}")
        df['_is_wrong'] = df['_is_wrong'] & ~df['new_std_nm'].isin(vanishing)

    # drop 목록 출력 (확인용)
    wrong_summary = (
        df[df['_is_wrong']]
        .groupby(['prdlst_no', '_mode_item', 'new_std_nm'])
        .size()
        .reset_index(name='건수')
        .rename(columns={'_mode_item': '정답품목', 'new_std_nm': '오입력품목'})
        .sort_values(['prdlst_no', '건수'], ascending=[True, False])
    )
    print(f"  drop 예정: {df['_is_wrong'].sum():,}건 ({len(wrong_summary)}종)")
    if not wrong_summary.empty:
        print(wrong_summary.to_string(index=False))

    return df  # _is_wrong 칼럼 포함한 상태로 반환 (drop 미실행)

df23 = detect_mapping_errors(df23, 'df23')
df24 = detect_mapping_errors(df24, 'df24')
df25 = detect_mapping_errors(df25, 'df25')

print("\n👆 위 목록 확인 후 drop 진행.")


[df23]
  결측치(기타) 처리: 194건 → 0건
  소멸 보호 품목: ['마늘', '소금', '파', '호박']
  drop 예정: 1,261건 (442종)
 prdlst_no 정답품목 오입력품목  건수
         4  복숭아     배   4
         4  복숭아    버섯   3
         4  복숭아    배추   1
         7    귤    국수   6
         7    귤    단감   3
         8  오렌지    양파  10
         8  오렌지     귤   3
         8  오렌지    단감   1
         8  오렌지    사과   1
         8  오렌지    설탕   1
         8  오렌지    소주   1
         8  오렌지    식초   1
         8  오렌지     쌀   1
         9   참외   오렌지   4
         9   참외    조기   4
         9   참외    설탕   1
         9   참외   소고기   1
         9   참외    소주   1
         9   참외   시금치   1
         9   참외    우유   1
         9   참외    조개   1
         9   참외   즉석밥   1
        10   수박    소주  10
        10   수박    참외   3
        10   수박    비누   1
        10   수박    상추   1
        10   수박    생수   1
        10   수박   오렌지   1
        11   딸기    두부   5
        11   딸기    된장   4
        11   딸기    수박   4
        11   딸기    당근   2
        11   딸기  돼지고기   1
        11   딸기    참외  

In [385]:
# 손실예정품목별 갯수 확인
# 1. 데이터셋 리스트와 연도 이름 설정
datasets = [df23, df24, df25]
years = ['2023년', '2024년', '2025년']

# 2. 손실된 품목 리스트 (아래 코드 확인 후 리스트 작성)
lost_items = {'고춧가루 1kg', '새우 1kg', '배 1개', '버섯 100g', '파프리카 100g'}

print("=== 연도별 손실 예정 품목 리스트업 ===\n")

for year_label, df in zip(years, datasets):
    print(f"[{year_label} 데이터 확인]")
    
    # 해당 데이터셋에 lost_items 중 존재하는 것들만 필터링
    counts = df[df['prdlst_nm'].isin(lost_items)]['prdlst_nm'].value_counts()
    
    if counts.empty:
        print(" -> 해당 연도에는 손실 예정 품목이 존재하지 않습니다.")
    else:
        for item, cnt in counts.items():
            print(f" - {item}: {cnt}개")
            
    print("-" * 35)

=== 연도별 손실 예정 품목 리스트업 ===

[2023년 데이터 확인]
 - 파프리카 100g: 21개
 - 새우 1kg: 20개
 - 배 1개: 19개
 - 버섯 100g: 16개
 - 고춧가루 1kg: 12개
-----------------------------------
[2024년 데이터 확인]
 -> 해당 연도에는 손실 예정 품목이 존재하지 않습니다.
-----------------------------------
[2025년 데이터 확인]
 -> 해당 연도에는 손실 예정 품목이 존재하지 않습니다.
-----------------------------------


In [386]:
# 맵핑 오류 목록 drop
def apply_drop(df, year_label):
    df = df[~df['_is_wrong']].copy()
    df.drop(columns=['_ratio', '_mode_item', '_is_wrong'], inplace=True)
    print(f"[{year_label}] 처리 후: {len(df):,}행")
    return df

df23 = apply_drop(df23, 'df23')
df24 = apply_drop(df24, 'df24')
df25 = apply_drop(df25, 'df25')

print("\n✅ 맵핑 오류 처리 완료")

[df23] 처리 후: 75,200행
[df24] 처리 후: 97,571행
[df25] 처리 후: 144,684행

✅ 맵핑 오류 처리 완료


In [387]:
# drop 후 확인
ori_after_drop_set = set(df23['prdlst_nm']) | set(df24['prdlst_nm']) | set(df25['prdlst_nm'])

# 손실된 품목 확인
print(f"  drop 후 전체 3개년도 고유 품목 수: {len(ori_after_drop_set)}개")

ori_lost = ori_all_unique_set - ori_after_drop_set
print(f"  drop으로 손실된 품목 수: {len(ori_lost)}개")
print(f"  손실된 품목: {ori_lost}")

  drop 후 전체 3개년도 고유 품목 수: 200개
  drop으로 손실된 품목 수: 5개
  손실된 품목: {'고춧가루 1kg', '새우 1kg', '배 1개', '버섯 100g', '파프리카 100g'}


In [388]:
# drop 후 확인
after_drop_set = set(df23['new_std_nm']) | set(df24['new_std_nm']) | set(df25['new_std_nm'])

# 손실된 품목 확인
print(f"  drop 후 전체 3개년도 고유 대표품목 수: {len(after_drop_set)}개")

lost = all_unique_set - after_drop_set
print(f"  drop으로 손실된 대표품목 수: {len(lost)}개")
print(f"  손실된 대표품목: {lost}")

  drop 후 전체 3개년도 고유 대표품목 수: 127개
  drop으로 손실된 대표품목 수: 1개
  손실된 대표품목: {'기타'}


#### 데이터 통합

In [389]:
df_total = pd.concat([df23, df24, df25], axis=0, ignore_index=True)

print(f"✅ 통합 완료: {df_total.shape[0]:,}행 × {df_total.shape[1]}열")
del df23, df24, df25
gc.collect()
print("   원본 데이터프레임 메모리 해제")

✅ 통합 완료: 317,455행 × 22열
   원본 데이터프레임 메모리 해제


In [390]:
print(df_total['prdlst_nm'].nunique(dropna=False))
df_total['prdlst_nm'].value_counts().sort_index()

200


prdlst_nm
 쌀(오대쌀) 10kg 1포     1128
 쌀(오대쌀) 20kg 1포      658
 쌀(오대쌀) 4kg 1포       934
 쌀(이천쌀) 10kg 1포     1119
 쌀(이천쌀) 20kg 1포     1597
 쌀(이천쌀) 4kg 1포       876
가지 1개                472
간장 1통               3756
갈치 1마리               858
갈치(냉동) 1마리(대)        217
갈치(냉동) 1마리(소)        102
갈치(냉동) 1마리(중)        305
갈치(생물) 1마리(대)       1916
갈치(생물) 1마리(소)        199
갈치(생물) 1마리(중)        723
감 10개                  1
감자 100g             3003
감자 1kg               957
갓 1kg                468
계란 10개              3512
고구마 1kg              950
고구마(밤고구마) 1kg       2969
고구마(호박고구마) 1kg      1528
고등어 1마리              864
고등어(신선냉장) 1마리(대)    1682
고등어(신선냉장) 1마리(중)     878
고등어(염장) 1손(대)       1089
고등어(염장) 1손(중)        756
고무장갑                  36
고추장 1kg             3646
고춧가루(국산) 1kg        3666
골드키위(1팩)             918
국수 900g             3752
굴 1kg                474
굵은소금(천일염) 1kg       3407
귤 10개                911
귤(제주산) 10개          2337
기저귀                    9
김치 3.3kg 1개          270
깐마늘 1kg        

In [391]:
print(df_total['new_std_nm'].nunique(dropna=False))
df_total['new_std_nm'].value_counts().sort_index()

127


new_std_nm
가지         472
간장        3761
갈치        4320
감            1
감자        3965
갓          468
계란        3512
고구마       5449
고등어       5270
고무장갑        36
고추장       3655
고춧가루      3667
골드키위       918
국수        3753
굴          474
굵은소금      3410
귤         3248
기저귀          9
김치         270
깐마늘       3847
깻잎        3848
꽃게         464
꽈리고추      1727
낙지         467
단감        3067
닭고기       3656
당근        3895
대추          81
대파        3955
도라지        468
돼지고기      3890
된장        3662
두부        3281
딸기        3153
라면        3830
마늘           5
마른멸치      3751
마요네즈      3803
만두        2814
맛김        2786
맥주        3559
멸치액젓       470
명태        3279
무         3961
물티슈         24
미나리        474
밀가루       3557
바나나       3674
바디워시      2811
밤           71
방울토마토     1702
배         3857
배추        3862
버섯        3843
복숭아       2407
부추         476
부침가루      3718
분유         816
붉은고추      1491
브로콜리      1834
비누        3386
빵         3046
사과        3868
사이다       3546
상추        5475
새우        2470

### 시장마트명 결측치 처리

In [392]:
# 1. mkplc_mart_no와 mkplc_mart_nm의 대응 관계를 사전(dict)으로 생성
# 결측치가 없는 행들만 추출하여 번호를 인덱스로, 이름을 값으로 설정
mapping_dict = df_total.dropna(subset=['mkplc_mart_nm']).set_index('mkplc_mart_no')['mkplc_mart_nm'].to_dict()

# 2. mkplc_mart_nm이 결측치인 경우, mkplc_mart_no를 기준으로 매핑 사전에서 값을 찾아 채움
df_total['mkplc_mart_nm'] = df_total['mkplc_mart_nm'].fillna(df_total['mkplc_mart_no'].map(mapping_dict))

print(f"✅ 시장마트번호 결측치: {df_total['mkplc_mart_nm'].isnull().sum():,}건")

✅ 시장마트번호 결측치: 0건


### 전통시장 데이터만 필터링

- 대형마트 18,807 건으로 불균형이 너무 심해서 전통시장 vs 대형마트 비교보다 전통시장 물가 현황 방향으로 진행

In [ ]:
df_total = df_total[df_total['mkplc_type_nm'] == '전통시장'].copy()
df_total['chck_ymd'] = pd.to_datetime(df_total['chck_ymd'])
df_total = df_total.sort_values('chck_ymd').reset_index(drop=True)

print(f"✅ 전통시장 필터링 후: {len(df_total):,}행")

✅ 전통시장 필터링 후: 317,455행


In [ ]:
# # 1. 전체 품목 집합 (전처리 전)
# all_items = set(df_total['prdlst_nm'].unique())

# # 2. 전처리 후 남은 품목 집합
# remaining_items = set(df_totala['prdlst_nm'].unique())

# # 3. 차집합을 통해 완전히 사라진 품목 식별
# lost_items = all_items - remaining_items

# print(f"📊 대형마트 필터링으로 인해 완전히 소실된 품목 수: {len(lost_items)}개")
# if lost_items:
#     print(f"🔎 소실된 품목 리스트: {lost_items}")
# else:
#     print("✅ 모든 품목이 최소 1개 이상의 정상 가격 데이터를 가지고 있어 소실된 품목명이 없습니다.")

📊 대형마트 필터링으로 인해 완전히 소실된 품목 수: 3개
🔎 소실된 품목 리스트: {'일회용마스크', '전지분유', '일회용접시'}


In [ ]:
# # 1. 전체 품목 집합 (전처리 전)
# all_items = set(df_total['new_std_nm'].unique())

# # 2. 전처리 후 남은 품목 집합
# remaining_items = set(df_total['new_std_nm'].unique())

# # 3. 차집합을 통해 완전히 사라진 품목 식별
# lost_items = all_items - remaining_items

# print(f"📊 대형마트 필터링으로 인해 완전히 소실된 품목 수: {len(lost_items)}개")
# if lost_items:
#     print(f"🔎 소실된 품목 리스트: {lost_items}")
# else:
#     print("✅ 모든 품목이 최소 1개 이상의 정상 가격 데이터를 가지고 있어 소실된 품목명이 없습니다.")

📊 대형마트 필터링으로 인해 완전히 소실된 품목 수: 3개
🔎 소실된 품목 리스트: {'일회용마스크', '전지분유', '일회용접시'}


### 시계열 파생변수 생성

In [ ]:
# 시계열 파생변수 (반기 / 분기 / 계절)
def get_season(month):
    if month in [3, 4, 5]:    return '봄'
    elif month in [6, 7, 8]:  return '여름'
    elif month in [9, 10, 11]: return '가을'
    else:                      return '겨울'

df_total['반기'] = df_total['chck_ymd'].dt.month.apply(lambda m: '상반기' if m <= 6 else '하반기')
df_total['분기'] = df_total['chck_ymd'].dt.quarter
df_total['계절'] = df_total['chck_ymd'].dt.month.apply(get_season)

print("✅ 시계열 파생변수 생성 완료")

✅ 시계열 파생변수 생성 완료


### 가격 관련 전처리

#### 보정 가격 계산(특정 품목)

In [ ]:
# 특정 품목 대표 규격 단위로 가격 보정
def calculate_standard_price(row):
    """규격별 가격을 단일 기준 단위 가격으로 보정"""
    item    = row['new_std_nm']
    variety = row['variety']
    spec    = row['spec']
    price   = row['pc']

    detailed_nm = f"{item}_{variety}" if variety != '기본' else item

    num_match = re.search(r'(\d+(?:\.\d+)?)', spec)
    amount = float(num_match.group(1)) if num_match else 1.0

    adj_price = price
    is_valid  = True

    if item == '쌀':
        adj_price = (price / amount) * 10 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    elif item in ['소고기', '돼지고기', '닭고기', '감자']:
        if 'kg' in spec:
            adj_price = (price / (amount * 1000)) * 100
        elif 'g' in spec:
            adj_price = (price / amount) * 100
        else:
            is_valid = False

    elif item in ['고등어', '갈치', '조기']:
        if '손' in spec:
            adj_price = price / (amount * 2)
        elif '마리' in spec:
            adj_price = price / amount
        else:
            is_valid = False

    elif item == '포도':
        adj_price = (price / amount) * 2 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    return detailed_nm, adj_price, is_valid

results = df_total.apply(calculate_standard_price, axis=1)
df_total['detailed_nm'] = [r[0] for r in results]
df_total['adj_price']   = [r[1] for r in results]
df_total['is_valid']    = [r[2] for r in results]

df_final = df_total[df_total['is_valid']].copy()
print(f"✅ 가격 보정 완료 → 유효 데이터: {len(df_final):,}행")

In [ ]:
# # ver2. 손실 방지 버전
# def calculate_standard_price(row):
#     """규격별 가격을 단일 기준 단위 가격으로 보정 (데이터 보존형)"""
#     item    = row['new_std_nm']
#     variety = row['variety']
#     spec    = str(row['spec'])  # NaN 방지를 위해 문자열 변환
#     price   = row['pc']

#     # 1. 상세 품목명 설정
#     detailed_nm = f"{item}_{variety}" if variety != '기본' else item

#     # 2. 숫자 추출 (예: '1.5kg' -> 1.5)
#     num_match = re.search(r'(\d+(?:\.\d+)?)', spec)
#     amount = float(num_match.group(1)) if num_match else 1.0

#     # 초기값 설정 (보정 규칙에 안 맞으면 원본 가격 유지)
#     adj_price = price
#     # logic_applied: 보정 로직이 실제로 작동했는지 여부 기록
#     logic_applied = False 

#     # 3. 보정 로직 (is_valid=False 대신 logic_applied로 상태 관리)
#     if item == '쌀':
#         if 'kg' in spec:
#             adj_price = (price / amount) * 10
#             logic_applied = True

#     elif item in ['소고기', '돼지고기', '닭고기', '감자']:
#         if 'kg' in spec:
#             adj_price = (price / (amount * 1000)) * 100
#             logic_applied = True
#         elif 'g' in spec:
#             adj_price = (price / amount) * 100
#             logic_applied = True

#     elif item in ['고등어', '갈치', '조기']:
#         if '손' in spec:
#             adj_price = price / (amount * 2)
#             logic_applied = True
#         elif '마리' in spec:
#             adj_price = price / amount
#             logic_applied = True

#     elif item == '포도':
#         if 'kg' in spec:
#             adj_price = (price / amount) * 2
#             logic_applied = True

#     return detailed_nm, adj_price, logic_applied

# # 4. 함수 적용
# results = df_total.apply(calculate_standard_price, axis=1)

# df_total['detailed_nm']  = [r[0] for r in results]
# df_total['adj_price']    = [r[1] for r in results]
# df_total['logic_applied'] = [r[2] for r in results]

# # 5. 최종 데이터셋 (가격이 0원인 쓰레기 데이터만 제거)
# # 이제 규격이 안 맞아도(logic_applied=False) 행을 삭제하지 않으므로 nunique가 유지됨
# df_final = df_total[df_total['adj_price'] > 0].copy()

# print(f"✅ 전처리 완료: {len(df_total):,} -> {len(df_final):,}행")
# print(f"✅ 고유 품목 수(nunique): {df_final['new_std_nm'].nunique()}개")

✅ 전처리 완료: 298,648 -> 254,965행
✅ 고유 품목 수(nunique): 124개


#### 가격 이상치 처리

In [ ]:
before = len(df_final)
df_clean = df_final[df_final['adj_price'] > 0].dropna(subset=['adj_price']).copy()

print(f"✅ 0 이하 제거: {before:,}건 → {len(df_clean):,}건  (제거: {before - len(df_clean):,}건)")

✅ 0 이하 제거: 254,965건 → 254,965건  (제거: 0건)


In [ ]:
# 1. 전체 품목 집합 (전처리 전)
all_items = set(df_final['new_std_nm'].unique())

# 2. 전처리 후 남은 품목 집합
remaining_items = set(df_clean['new_std_nm'].unique())

# 3. 차집합을 통해 완전히 사라진 품목 식별
lost_items = all_items - remaining_items

print(f"📊 가격 필터링으로 인해 완전히 소실된 품목 수: {len(lost_items)}개")
if lost_items:
    print(f"🔎 소실된 품목 리스트: {lost_items}")
else:
    print("✅ 모든 품목이 최소 1개 이상의 정상 가격 데이터를 가지고 있어 소실된 품목명이 없습니다.")

📊 가격 필터링으로 인해 완전히 소실된 품목 수: 0개
✅ 모든 품목이 최소 1개 이상의 정상 가격 데이터를 가지고 있어 소실된 품목명이 없습니다.


In [ ]:
print(df_clean['new_std_nm'].nunique())

124


- 오타로 인한 이상치 처리

In [ ]:
# 휴먼 에러로 판단되는 이상치 보정
def auto_correct_price_typos(df):
    """중앙값 대비 비율 기반 자릿수 오타 자동 보정"""
    df_c = df.copy()
    median_dict = df_c.groupby('detailed_nm')['adj_price'].median().to_dict()

    def correct_logic(row):
        price  = row['adj_price']
        median = median_dict.get(row['detailed_nm'], price)
        if median == 0 or pd.isna(price):
            return price, '유지(계산불가)'
        ratio = price / median

        if   0.05 <= ratio <= 0.15:  return price * 10,  '×10 보정'
        elif 0.005 <= ratio <= 0.02: return price * 100, '×100 보정'
        elif 5.0 <= ratio <= 15.0:   return price / 10,  '÷10 보정'
        elif 50.0 <= ratio <= 150.0: return price / 100, '÷100 보정'
        else:                         return price,        '유지'

    results = df_c.apply(correct_logic, axis=1)
    df_c['adj_price']       = [r[0] for r in results]
    df_c['correction_type'] = [r[1] for r in results]
    return df_c

df_clean = auto_correct_price_typos(df_clean)

print("✅ 자릿수 오타 보정 결과")
print(df_clean['correction_type'].value_counts().to_string())

✅ 자릿수 오타 보정 결과
correction_type
유지         253162
÷10 보정       1519
×10 보정        283
÷100 보정         1


- IQR 기준 이상치 처리 (품목별 분류 체계 추가 후 진행)
    - 생성된 분류체계(대/중/소) 기준, 가격변동성 커도 정상인 항목과 가격변동성이 크지 않다고 판단되는 분류 구분
    - 가격변동성 크지 않다고 분류한 대상 분류에 대해서만 진행

In [ ]:
# # IQR 기반 잔존 이상치 제거
# def remove_iqr_outliers(df):
#     """상세품목명 기준 IQR 이탈 데이터 제거 (중앙값 ±20% 최소 버퍼 적용)"""
#     stats_df = df.groupby('detailed_nm', as_index=False).agg(
#         median=('adj_price', 'median'),
#         q1    =('adj_price', lambda x: x.quantile(0.25)),
#         q3    =('adj_price', lambda x: x.quantile(0.75)),
#     )
#     stats_df['iqr']     = stats_df['q3'] - stats_df['q1']
#     stats_df['lower_b'] = stats_df['q1'] - np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)
#     stats_df['upper_b'] = stats_df['q3'] + np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)

#     df_m     = df.merge(stats_df[['detailed_nm', 'lower_b', 'upper_b']], on='detailed_nm', how='left')
#     cond_iqr = (df_m['adj_price'] < df_m['lower_b']) | (df_m['adj_price'] > df_m['upper_b'])
#     df_out   = df_m[~cond_iqr].drop(columns=['lower_b', 'upper_b'])

#     print(f"✅ IQR 이상치 제거: {len(df_m):,}건 → {len(df_out):,}건  (제거: {cond_iqr.sum():,}건)")
#     return df_out

# df_clean = remove_iqr_outliers(df_clean)

### 품목별 대 / 중 / 소분류 체계 추가

> 📋 품목 분류 참고: [품목_분류_체계.md](../품목_분류_체계.md)

In [ ]:
# 품목 분류 체계 (대분류 / 중분류 / 소분류)
category_dict = {
    # ── 농축수산물 ─────────────────────────────────────────────
    '쌀':('농축수산물','농산물','곡류'), '콩':('농축수산물','농산물','곡류'),
    '팥':('농축수산물','농산물','곡류'),
    '배추':('농축수산물','농산물','채소'), '무':('농축수산물','농산물','채소'),
    '상추':('농축수산물','농산물','채소'), '대파':('농축수산물','농산물','채소'),
    '양파':('농축수산물','농산물','채소'), '오이':('농축수산물','농산물','채소'),
    '애호박':('농축수산물','농산물','채소'), '깻잎':('농축수산물','농산물','채소'),
    '당근':('농축수산물','농산물','채소'), '파프리카':('농축수산물','농산물','채소'),
    '시금치':('농축수산물','농산물','채소'), '토마토':('농축수산물','농산물','채소'),
    '깐마늘':('농축수산물','농산물','채소'), '마늘':('농축수산물','농산물','채소'),
    '감자':('농축수산물','농산물','채소'), '고구마':('농축수산물','농산물','채소'),
    '콩나물':('농축수산물','농산물','채소'), '버섯':('농축수산물','농산물','채소'),
    '풋고추':('농축수산물','농산물','채소'), '청양고추':('농축수산물','농산물','채소'),
    '꽈리고추':('농축수산물','농산물','채소'), '붉은고추':('농축수산물','농산물','채소'),
    '양배추':('농축수산물','농산물','채소'), '방울토마토':('농축수산물','농산물','채소'),
    '브로콜리':('농축수산물','농산물','채소'), '부추':('농축수산물','농산물','채소'),
    '쪽파':('농축수산물','농산물','채소'), '가지':('농축수산물','농산물','채소'),
    '생강':('농축수산물','농산물','채소'), '미나리':('농축수산물','농산물','채소'),
    '도라지':('농축수산물','농산물','채소'), '갓':('농축수산물','농산물','채소'),
    '열무':('농축수산물','농산물','채소'), '호박':('농축수산물','농산물','채소'),
    '파':('농축수산물','농산물','채소'),
    '사과':('농축수산물','농산물','과일'), '배':('농축수산물','농산물','과일'),
    '바나나':('농축수산물','농산물','과일'), '참외':('농축수산물','농산물','과일'),
    '수박':('농축수산물','농산물','과일'), '오렌지':('농축수산물','농산물','과일'),
    '귤':('농축수산물','농산물','과일'), '단감':('농축수산물','농산물','과일'),
    '감':('농축수산물','농산물','과일'), '포도':('농축수산물','농산물','과일'),
    '딸기':('농축수산물','농산물','과일'), '골드키위':('농축수산물','농산물','과일'),
    '복숭아':('농축수산물','농산물','과일'), '대추':('농축수산물','농산물','과일'),
    '밤':('농축수산물','농산물','과일'),
    '소고기':('농축수산물','축산물','정육'), '돼지고기':('농축수산물','축산물','정육'),
    '닭고기':('농축수산물','축산물','정육'), '계란':('농축수산물','축산물','알류'),
    '고등어':('농축수산물','수산물','생선류'), '갈치':('농축수산물','수산물','생선류'),
    '명태':('농축수산물','수산물','생선류'), '조기':('농축수산물','수산물','생선류'),
    '오징어':('농축수산물','수산물','해산물'), '새우':('농축수산물','수산물','해산물'),
    '조개':('농축수산물','수산물','해산물'), '굴':('농축수산물','수산물','해산물'),
    '낙지':('농축수산물','수산물','해산물'), '전복':('농축수산물','수산물','해산물'),
    '꽃게':('농축수산물','수산물','해산물'),
    '마른멸치':('농축수산물','수산물','건어물/해조류'),
    '맛김':('농축수산물','수산물','건어물/해조류'),
    # ── 가공식품 ───────────────────────────────────────────────
    '설탕':('가공식품','조미료','소스/오일'), '식초':('가공식품','조미료','소스/오일'),
    '식용유':('가공식품','조미료','소스/오일'), '간장':('가공식품','조미료','소스/오일'),
    '마요네즈':('가공식품','조미료','소스/오일'), '케찹':('가공식품','조미료','소스/오일'),
    '참기름':('가공식품','조미료','소스/오일'),
    '새우젓':('가공식품','조미료','젓갈류'), '멸치액젓':('가공식품','조미료','젓갈류'),
    '고춧가루':('가공식품','조미료','가루/장류'), '된장':('가공식품','조미료','가루/장류'),
    '고추장':('가공식품','조미료','가루/장류'), '밀가루':('가공식품','조미료','가루/장류'),
    '부침가루':('가공식품','조미료','가루/장류'),
    '굵은소금':('가공식품','조미료','가루/장류'), '소금':('가공식품','조미료','가루/장류'),
    '라면':('가공식품','면/빵류','면류'), '컵라면':('가공식품','면/빵류','면류'),
    '국수':('가공식품','면/빵류','면류'), '빵':('가공식품','면/빵류','빵류'),
    '통조림':('가공식품','간편식','간편조리'), '즉석밥':('가공식품','간편식','간편조리'),
    '어묵':('가공식품','간편식','간편조리'), '만두':('가공식품','간편식','간편조리'),
    '김치':('가공식품','간편식','간편조리'),
    '햄':('가공식품','간편식','가공육'), '소시지':('가공식품','간편식','가공육'),
    '두부':('가공식품','간편식','두부류'),
    '우유':('가공식품','유제품/간식','유제품'), '치즈':('가공식품','유제품/간식','유제품'),
    '분유':('가공식품','유제품/간식','유제품'), '전지분유':('가공식품','유제품/간식','유제품'),
    '에너지바':('가공식품','유제품/간식','간식류'),
    '초콜릿':('가공식품','유제품/간식','간식류'), '캔디':('가공식품','유제품/간식','간식류'),
    # ── 음료/주류 ──────────────────────────────────────────────
    '사이다':('음료/주류','음료','탄산/생수'), '콜라':('음료/주류','음료','탄산/생수'),
    '생수':('음료/주류','음료','탄산/생수'),
    '소주':('음료/주류','주류','주류'), '맥주':('음료/주류','주류','주류'),
    # ── 생필품 ────────────────────────────────────────────────
    '비누':('생필품','위생용품','바디/헤어'), '샴푸':('생필품','위생용품','바디/헤어'),
    '바디워시':('생필품','위생용품','바디/헤어'),
    '칫솔':('생필품','위생용품','구강용품'), '치약':('생필품','위생용품','구강용품'),
    '세제':('생필품','생활잡화','세제/세정'), '주방세제':('생필품','생활잡화','세제/세정'),
    '섬유유연제':('생필품','생활잡화','세제/세정'),
    '위생백':('생필품','생활잡화','주방잡화'), '고무장갑':('생필품','생활잡화','주방잡화'),
    '일회용컵':('생필품','생활잡화','주방잡화'), '일회용접시':('생필품','생활잡화','주방잡화'),
    '일회용마스크':('생필품','위생용품','위생용품'),
    '휴지':('생필품','위생용품','지류/물티슈'), '물티슈':('생필품','위생용품','지류/물티슈'),
    '기저귀':('생필품','위생용품','위생용품'), '여성용품':('생필품','위생용품','위생용품'),
}

df_category = (
    pd.DataFrame.from_dict(category_dict, orient='index',
                            columns=['대분류', '중분류', '소분류'])
    .reset_index()
    .rename(columns={'index': 'base_nm'})
)

df_clean['base_nm'] = df_clean['detailed_nm'].str.split('_').str[0]
df_clean = df_clean.merge(df_category, on='base_nm', how='left')
df_clean[['대분류', '중분류', '소분류']] = df_clean[['대분류', '중분류', '소분류']].fillna('기타')
df_clean.drop(columns=['base_nm'], inplace=True)

print("✅ 분류 체계 매핑 완료")
print(df_clean.groupby('대분류').size().rename('건수').to_string())

✅ 분류 체계 매핑 완료
대분류
가공식품      72806
농축수산물    155974
생필품       10328
음료/주류     15857


### 불필요 칼럼 제거 및 칼럼명 한글로 정리

In [ ]:
# 불필요 칼럼 제거 및 칼럼명 한글화
drop_cols = ['is_valid', 'correction_type']
drop_cols = [c for c in drop_cols if c in df_clean.columns]
df_clean.drop(columns=drop_cols, inplace=True)

re_col_mapping = {
    'sn':              '일련번호',
    'mkplc_mart_no':   '시장마트번호',
    'mkplc_mart_nm':   '시장마트명',
    'prdlst_no':       '품목번호',
    'prdlst_nm':       '품목명',
    'real_sle_stndrd': '실제판매규격',
    'pc':              '가격',
    'ym':              '연월',
    'rmrk':            '비고',
    'mkplc_type_cd':   '시장유형코드',
    'mkplc_type_nm':   '시장유형명',
    'atdrc_cd':        '자치구코드',
    'atdrc':           '자치구',
    'chck_ymd':        '점검일자',
    'new_std_nm':      '대표품목명',
    'detailed_nm':     '상세품목명',
    'adj_price':       '보정가격',
}

rename_map = {k: v for k, v in re_col_mapping.items() if k in df_clean.columns}
df_clean.rename(columns=rename_map, inplace=True)

print("✅ 전처리 완료")
print(f"   최종 데이터: {df_clean.shape[0]:,}행 × {df_clean.shape[1]}열")
print(f"   분석 기간  : {df_clean['점검일자'].min().strftime('%Y-%m-%d')} ~ {df_clean['점검일자'].max().strftime('%Y-%m-%d')}")
print(f"   자치구 수  : {df_clean['자치구'].nunique()}개")
print(f"   대표 품목  : {df_clean['대표품목명'].nunique()}개")

✅ 전처리 완료
   최종 데이터: 254,965행 × 31열
   분석 기간  : 2023-03-27 ~ 2026-04-13
   자치구 수  : 25개
   대표 품목  : 124개


In [ ]:
print(df_clean.isnull().sum())

일련번호                  0
시장마트번호                0
시장마트명                 0
품목번호                  0
품목명                 257
실제판매규격           254965
가격                    0
연월                    0
비고               225382
시장유형코드              849
시장유형명                 0
자치구코드                 0
자치구                   0
점검일자                  0
대표품목명                 0
variety               0
spec                  0
prdlst_cd        129259
prdlst_std_nm    129259
vrty_nm          129259
unit             150193
qty_nm           172013
반기                    0
분기                    0
계절                    0
상세품목명                 0
보정가격                  0
logic_applied         0
대분류                   0
중분류                   0
소분류                   0
dtype: int64
